# 06 · Newsvendor buffer (Phase 6) — headline output

| Newsvendor | Our project |
|---|---|
| Order quantity *Q* | buffer *b* added to the quoted COE price |
| Uncertain demand *D* | premium rise *R* between quote and purchase |
| Underage cost *c_u* | *R* > *b*: dealer absorbs the shortfall, *c_u* = S$1 per S$1 short |
| Overage cost *c_o* | *b* > *R*: car overpriced → lost or discounted sales; not public |
| Critical ratio *c_u/(c_u+c_o)* | percentile τ of the forecast distribution to charge |

$$b^* = \max\bigl(0,\; F_R^{-1}(\tau^*)\bigr), \qquad \tau^* = \frac{k}{1+k}, \quad k = c_u/c_o$$

Because *c_o* is unknown we sweep τ = 0.5 … 0.95 (k = 1 … 19, "underpricing costs k times as much as overpricing").

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import *
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
from src.newsvendor import tau_to_k, implied_cost_ratios, condition_cutoffs, buffer_table, buffer_sgd, assign_condition
fc = pd.read_csv(PROC / "forecasts.csv")
feat = pd.read_csv(PROC / "coe_features.csv")
best = json.load(open(PROC / "best_model.json"))["best_model"]
print("Buffer model:", best)
pd.DataFrame({"tau": TAUS, "k = c_u / c_o": [round(tau_to_k(t), 1) for t in TAUS],
              "meaning": [f"underpricing costs {tau_to_k(t):.1f}x overpricing" for t in TAUS]})

Buffer model: M1


,tau,k = c_u / c_o,meaning
0,0.500,1.000,underpricing costs 1.0x overpricing
1,0.600,1.500,underpricing costs 1.5x overpricing
2,0.700,2.300,underpricing costs 2.3x overpricing
3,0.800,4.000,underpricing costs 4.0x overpricing
4,0.900,9.000,underpricing costs 9.0x overpricing
5,0.950,19.000,underpricing costs 19.0x overpricing


## Cost ratios implied by what dealers do today (h = 2, 2014–2026)

In [3]:
base = fc[(fc.model == "M0") & (fc.h == H) & (fc.tau == 0.5)].dropna(subset=["actual_sgd"])
imp = implied_cost_ratios(base)
display(imp.round(2))
imp_p = pd.concat([implied_cost_ratios(g).assign(period=p) for p, g in base.groupby("period")])
display(imp_p[imp_p.category == "All"].pivot(index="rule", columns="period", values="implied_tau").round(2))
imp.to_csv(PROC / "implied_cost_ratios.csv", index=False)
a = imp[imp.category == "All"].set_index("rule")
print(f"No buffer covers {a.loc['R0 No buffer','implied_tau']:.0%} of sales -> acts as if k = {a.loc['R0 No buffer','implied_k']:.2f} "
      f"(overpricing treated as the bigger risk).")
print(f"Fixed 5% covers {a.loc['R1 Fixed 5%','implied_tau']:.0%} -> acts as if k = {a.loc['R1 Fixed 5%','implied_k']:.2f}.")

,category,rule,implied_tau,implied_k
0,All,R0 No buffer,0.460,0.840
1,All,R1 Fixed 5%,0.720,2.600
2,A,R0 No buffer,0.450,0.820
3,A,R1 Fixed 5%,0.740,2.820
4,B,R0 No buffer,0.460,0.860
5,B,R1 Fixed 5%,0.700,2.390
6,E,R0 No buffer,0.450,0.830
7,E,R1 Fixed 5%,0.720,2.630


period,2014-19,2020-22,2023-26
rule,,,
R0 No buffer,0.550,0.380,0.360
R1 Fixed 5%,0.780,0.590,0.720


No buffer covers 46% of sales -> acts as if k = 0.84 (overpricing treated as the bigger risk).
Fixed 5% covers 72% -> acts as if k = 2.60.


## The dealer's buffer table

Rows: category × market condition = **recent volatility** at the quote round (`vol_12`, the std. dev. of 1-round % changes over the last 12 rounds; terciles per category). Volatility is used rather than the bid ratio because the EDA and the driver regression found that the bid ratio barely predicts the next rise, while volatility clusters (calm and turbulent spells persist). The bid-ratio version is kept below for comparison. Columns: cost ratio *k*. Each cell: the median newsvendor buffer from the rounds in that cell (2014–2026, out-of-sample) as % of the premium, applied to today's premium (Oct 2026 round 2). The backtest columns use the actual per-round buffers.

In [4]:
cuts = condition_cutoffs(feat, "vol_12")
latest = feat.sort_values("t").groupby("category").premium.last().to_dict()
print("volatility tercile cut-offs (std. dev. of 1-round changes):", {c: tuple(f"{x:.1%}" for x in v) for c, v in cuts.items()})
print("latest premiums:", latest)
fb = fc[(fc.model == best) & (fc.h == H)].merge(feat[["category", "t", "vol_12"]], on=["category", "t"])
bt_table = buffer_table(fb, cuts, latest, "vol_12")
bt_table.to_csv(PROC / "buffer_table.csv", index=False)
cell = bt_table.assign(cell=lambda d: d.apply(lambda r: f"S${r.buffer_sgd:,.0f} ({r.buffer_pct:.1f}%)", axis=1))
pretty = cell.pivot_table(index=["category", "condition", "signal_range"], columns="k", values="cell", aggfunc="first")
pretty = pretty.reindex(sorted(pretty.index, key=lambda i: (i[0], {"Low": 0, "Medium": 1, "High": 2}[i[1]])))
pretty.columns = [f"k={c}" for c in pretty.columns]
pretty.to_csv(PROC / "buffer_table_pretty.csv")
pretty

volatility tercile cut-offs (std. dev. of 1-round changes): {'A': ('4.7%', '7.1%'), 'B': ('4.9%', '7.1%'), 'E': ('4.1%', '5.5%')}
latest premiums: {'A': 132004, 'B': 131901, 'E': 135790}


k=1.0           k=1.5           k=2.3            k=4.0             k=9.0            k=19.0
category condition signal_range                                                                                                     
A        Low       <= 4.7%       S$2,433 (1.8%)  S$4,043 (3.1%)  S$6,437 (4.9%)   S$7,912 (6.0%)   S$10,629 (8.1%)  S$14,755 (11.2%)
         Medium    4.7%-7.1%     S$2,187 (1.7%)  S$4,245 (3.2%)  S$6,665 (5.0%)   S$9,242 (7.0%)  S$13,227 (10.0%)  S$16,738 (12.7%)
         High      > 7.1%            S$0 (0.0%)  S$1,189 (0.9%)  S$3,015 (2.3%)   S$5,624 (4.3%)  S$13,538 (10.3%)  S$16,738 (12.7%)
B        Low       <= 4.9%         S$199 (0.2%)  S$2,609 (2.0%)  S$6,155 (4.7%)   S$8,635 (6.5%)   S$12,095 (9.2%)  S$20,041 (15.2%)
         Medium    4.9%-7.1%       S$541 (0.4%)  S$4,187 (3.2%)  S$6,168 (4.7%)   S$9,424 (7.1%)  S$15,602 (11.8%)  S$20,566 (15.6%)
         High      > 7.1%            S$0 (0.0%)  S$3,772 (2.9%)  S$5,853 (4.4%)  S$11,230 (8.5%)  S$14,264 (10.8%)  S$20,566 (15.6%)
E        Low       <= 4.1%       S$1,036 (0.8%)  S$3,482 (2.6%)  S$5,642 (4.2%)   S$8,726 (6.4%)   S$12,030 (8.9%)  S$14,268 (10.5%)
         Medium    4.1%-5.5%     S$1,132 (0.8%)  S$2,931 (2.2%)  S$4,954 (3.6%)   S$8,621 (6.3%)   S$11,964 (8.8%)  S$14,268 (10.5%)
         High      > 5.5%          S$251 (0.2%)  S$2,804 (2.1%)  S$5,729 (4.2%)   S$9,568 (7.0%)  S$13,611 (10.0%)  S$20,004 (14.7%)

In [5]:
print("Share of sales underwater with that buffer (2014-2026 backtest), %:")
uw = bt_table.pivot_table(index=["category", "condition"], columns="k", values="underwater_pct")
display(uw.reindex(sorted(uw.index, key=lambda i: (i[0], {"Low": 0, "Medium": 1, "High": 2}[i[1]]))).round(1))

Share of sales underwater with that buffer (2014-2026 backtest), %:


k                   1.000   1.500   2.300   4.000   9.000   19.000
category condition                                                
A        Low        46.300  37.200  26.400  16.500   6.600   4.100
         Medium     47.800  40.200  28.300  18.500  10.900   4.300
         High       44.700  40.000  32.900  23.500  14.100   7.100
B        Low        50.000  42.100  29.400  19.800  10.300   5.600
         Medium     40.000  30.000  21.000  12.000   5.000   3.000
         High       55.600  45.800  36.100  29.200  18.100  11.100
E        Low        51.400  47.700  43.200  28.800  15.300   9.000
         Medium     43.000  32.500  23.700  14.000   7.900   3.500
         High       50.700  38.400  23.300  13.700   6.800   4.100

In [6]:
r = bt_table[(bt_table.category == "B") & (bt_table.condition == "High") & (bt_table.tau == 0.8)].iloc[0]
c = bt_table[(bt_table.category == "B") & (bt_table.condition == "Low") & (bt_table.tau == 0.8)].iloc[0]
guidance = (f"A dealer who rates underpricing four times as costly as overpricing (k = 4, τ = 0.8) should add about "
            f"{r.buffer_pct:.1f}% (S${r.buffer_sgd:,.0f} at today's premium) to a Cat B package after a turbulent spell "
            f"(1-round changes over the last 12 rounds with a std. dev. above {cuts['B'][1]:.1%}), against {c.buffer_pct:.1f}% "
            f"(S${c.buffer_sgd:,.0f}) in a calm market (below {cuts['B'][0]:.1%}); in the 2014-2026 backtest those buffers "
            f"were too small on {r.underwater_pct:.0f}% and {c.underwater_pct:.0f}% of such sales.")
print(guidance)
(REPORT / "buffer_guidance.txt").write_text(guidance)

A dealer who rates underpricing four times as costly as overpricing (k = 4, τ = 0.8) should add about 8.5% (S$11,230 at today's premium) to a Cat B package after a turbulent spell (1-round changes over the last 12 rounds with a std. dev. above 7.1%), against 6.5% (S$8,635) in a calm market (below 4.9%); in the 2014-2026 backtest those buffers were too small on 29% and 20% of such sales.


389

## Recent regime (2023–2026 only)

Premiums and volatility are higher since 2023, so the same τ needs a bigger buffer. A dealer pricing today should look at this version too.

In [7]:
bt_recent = buffer_table(fb[fb.period == "2023-26"], cuts, latest)
bt_recent.to_csv(PROC / "buffer_table_2023_26.csv", index=False)
bt_recent.pivot_table(index=["category", "condition"], columns="k", values="buffer_pct").round(1)

k                   1.000   1.500   2.300   4.000   9.000   19.000
category condition                                                
A        High        2.400   4.600   5.200   7.800  10.500  11.400
         Low         2.300   3.200   4.900   5.600   8.500  10.600
         Medium      2.400   3.400   5.000   6.100   9.400  12.400
B        High        2.400   3.800   6.500   8.000   9.600  15.400
         Low         2.200   3.800   5.800   6.800   9.200  15.200
         Medium      2.200   3.800   5.100   6.000   8.700  15.200
E        High        0.600   4.200   7.200   9.200  10.500  11.000
         Low         1.300   2.800   4.300   7.000   8.800  10.400
         Medium      1.500   2.800   5.500   6.700   8.500  10.000

## Which market signal moves the buffer? Volatility vs the bid ratio

The same table cut by bid-ratio terciles (the original plan's condition). The spread is the largest minus the smallest buffer across the three conditions, in % points of the premium: a signal that does not change the buffer is no use to a dealer.

In [8]:
bcuts = condition_cutoffs(feat, "bid_ratio")
bt_bid = buffer_table(fb, bcuts, latest, "bid_ratio")
bt_bid.to_csv(PROC / "buffer_table_by_bidratio.csv", index=False)
print("Buffer % of premium by bid-ratio condition:")
display(bt_bid.pivot_table(index=["category", "condition"], columns="k", values="buffer_pct").round(1))
def spread(t):
    p = t.pivot_table(index=["category", "k"], columns="condition", values="buffer_pct")
    return p.max(axis=1) - p.min(axis=1)
comp = pd.DataFrame({"spread by volatility (pp)": spread(bt_table), "spread by bid ratio (pp)": spread(bt_bid)})
comp_k = comp.reset_index().pivot(index="k", columns="category")
comp_k.round(1)

Buffer % of premium by bid-ratio condition:


k                   1.000   1.500   2.300   4.000   9.000   19.000
category condition                                                
A        High        1.600   3.100   4.700   6.300   9.800  11.600
         Low         0.200   1.800   3.300   5.000   9.500  12.400
         Medium      1.700   3.100   4.900   6.400  10.300  12.700
B        High        0.000   2.100   4.700   6.900  10.000  15.600
         Low         0.000   1.600   4.000   7.100  11.300  15.600
         Medium      0.600   3.400   4.700   6.800   9.600  15.400
E        High        1.100   2.800   4.300   6.700   9.400  10.500
         Low         0.100   1.900   3.200   6.400   9.400  11.800
         Medium      0.200   2.200   3.600   6.400   9.400  11.300

spread by volatility (pp)             spread by bid ratio (pp)            
category                         A     B     E                        A     B     E
k                                                                                  
1.000                        1.800 0.400 0.600                    1.400 0.600 1.100
1.500                        2.300 1.200 0.500                    1.300 1.700 0.900
2.300                        2.800 0.200 0.600                    1.500 0.600 1.100
4.000                        2.700 2.000 0.700                    1.400 0.300 0.300
9.000                        2.200 2.700 1.200                    0.800 1.700 0.000
19.000                       1.500 0.400 4.200                    1.100 0.200 1.300

In [9]:
sign = bt_table.pivot_table(index=["category", "k"], columns="condition", values="buffer_pct")
mono = ((sign["Low"] <= sign["Medium"]) & (sign["Medium"] <= sign["High"])).groupby("category").mean()
sign_b = bt_bid.pivot_table(index=["category", "k"], columns="condition", values="buffer_pct")
mono_b = ((sign_b["Low"] <= sign_b["Medium"]) & (sign_b["Medium"] <= sign_b["High"])).groupby("category").mean()
print("Share of k values where the buffer rises Low <= Medium <= High:")
print(pd.DataFrame({"volatility": mono, "bid ratio": mono_b}).round(2).to_string())
print(f"\nMean spread across conditions: volatility {comp.iloc[:, 0].mean():.1f} pp vs bid ratio {comp.iloc[:, 1].mean():.1f} pp")

Share of k values where the buffer rises Low <= Medium <= High:
          volatility  bid ratio
category                       
A              0.330      0.000
B              0.330      0.170
E              0.170      0.670

Mean spread across conditions: volatility 1.6 pp vs bid ratio 1.0 pp


## Today's recommendation (forecast made at the last round, October 2026 round 2, for a purchase two rounds later)

In [10]:
last = fb[fb.round_id == fb.round_id.max()].copy()
last["buffer_sgd"] = buffer_sgd(last.q_pct, last.premium_now)
last["k"] = last.tau.map(lambda t: round(tau_to_k(t), 1))
last["condition"] = assign_condition(last, cuts, "vol_12")
now = last.pivot_table(index=["category", "premium_now", "vol_12", "bid_ratio", "condition"], columns="k", values="buffer_sgd")
now.columns = [f"k={c}" for c in now.columns]
now.to_csv(PROC / "buffer_today.csv")
now.round(0)

,,,,,k=1.0,k=1.5,k=2.3,k=4.0,k=9.0,k=19.0
category,premium_now,vol_12,bid_ratio,condition,,,,,,
A,132004,0.023,1.322,Low,"2,322.000","3,356.000","5,468.000","7,127.000","10,380.000","12,729.000"
B,131901,0.028,1.245,Low,"2,239.000","4,280.000","5,717.000","7,617.000","9,548.000","13,263.000"
E,135790,0.013,1.522,Low,"1,633.000","2,503.000","5,222.000","6,246.000","7,711.000","11,375.000"


## Optional: scenarios for the overage cost *c_o* (labelled assumptions, not estimates)

$c_o \approx$ (chance of losing the sale per extra S$1,000) × (dealer margin per car) ÷ 1,000. With *c_u* = 1, k = 1/c_o.

In [11]:
sc = pd.DataFrame({"scenario": ["Low c_o (loyal buyers)", "Medium", "High c_o (price-sensitive buyers)"],
                   "p_lose_per_1000": [0.02, 0.05, 0.10], "margin_per_car": [5000, 8000, 10000]})
sc["c_o"] = sc.p_lose_per_1000 * sc.margin_per_car / 1000
sc["k"] = 1 / sc.c_o; sc["tau_star"] = sc.k / (1 + sc.k)
sc["nearest_tau"] = sc.tau_star.map(lambda t: min(TAUS, key=lambda x: abs(x - t)))
sc

,scenario,p_lose_per_1000,margin_per_car,c_o,k,tau_star,nearest_tau
0,Low c_o (loyal buyers),0.020,5000,0.100,10.000,0.909,0.900
1,Medium,0.050,8000,0.400,2.500,0.714,0.700
2,High c_o (price-sensitive buyers),0.100,10000,1.000,1.000,0.500,0.500
